[Open in Google Colab](https://colab.research.google.com/github/kshamesh/dsa-industry-lens/blob/main/solutions/DSA_Instructor_Solutions.ipynb)

Save a personal copy before editing. Instructor solutions: attempt the student lab before reading these answers.


# DSA Through an Industry Lens
## Instructor solutions
MCA First Year · Python 3 · 60 minutes

**Goal:** explain a correct solution, identify repeated work, improve it, and test it.

Open this notebook using the Google Colab link above, or use Jupyter Notebook, JupyterLab or a configured VS Code notebook editor. Select a Python 3 kernel. No third-party Python packages or data downloads are required. Run cells from top to bottom. Save your own copy.

Pair up if helpful: one student codes, the other explains and tests. Swap roles between exercises.

A failed assertion means a result differs from the contract. Read the message, trace a small example, and try again.


## Schedule
| Minutes | Activity |
|---|---|
| 00–08 | Python warm-up |
| 08–23 | Budget pair |
| 23–38 | Two pointers |
| 38–53 | Sliding window |
| 53–60 | Test and explain |

**For every problem:** clarify inputs and outputs; try an example; propose a simple solution; find repeated work; improve; test; explain time and extra space.

Use indices starting at zero. Prices use whole currency units. These are teaching models, not production money-handling code.


## Delivery guide
Prepare the Python environment before the lab. Project each contract and let students predict output before coding. Give hints one at a time. If students fall behind, demonstrate two pointers and retain independent time for the window exercise.

Do not require every student to finish every optimization. A student who completes a correct baseline and explains it has made useful progress. Keep this notebook separate from the student copy until the debrief.

Classroom challenge answer: `[4,9,3,12,2]`, `k=2` gives totals 13,12,15,14. Return `(2,15)`, representing hours 3–4. Choosing 9 and 12 violates contiguity.


## 0. Python warm-up (8 minutes)
A college receives course registrations. Count the registrations for each course.

**Contract:** input is a list of course-name strings. Return a dictionary from course name to count. Matching is case-sensitive. Do not modify the input. Empty input returns `{}`.

`["Python", "Java", "Python", "DSA", "Python"]` should return `{"Python": 3, "Java": 1, "DSA": 1}`.

Why would a set lose information here?


In [ ]:
courses = ["Python", "Java", "Python", "DSA", "Python"]
print(len(courses))
for index, course in enumerate(courses):
    print(index, course)
print({"Python": 3}.get("Java", 0))  # default when key is absent


In [ ]:
def count_courses(courses):
    counts = {}
    for course in courses:
        counts[course] = counts.get(course, 0) + 1
    return counts


In [ ]:
assert count_courses([]) == {}
assert count_courses(["Python", "Java", "Python"]) == {"Python": 2, "Java": 1}
assert count_courses(["python", "Python"]) == {"python": 1, "Python": 1}
print("Warm-up checks passed")


<details><summary>Hints</summary>

1. Start with an empty dictionary.
2. Visit each course.
3. Read its current count (default zero), add one, and store it.
</details>


**Explain:** a set tells us which courses exist, but discards frequency. The loop maintains counts for the prefix already visited. Expected O(n) time, O(u) extra space for u distinct courses. Common errors: resetting a count to 1 each time or reading a missing key without a default.


## 1. Two prices, one budget (15 minutes)
**Contract:** input is a list of nonnegative integer prices and an integer budget. Return any tuple of two distinct indices whose prices sum to the budget, or `None` when no pair exists. Values can repeat. Do not mutate the input.

`[120,80,160,40]`, budget `200`: `(0,1)` and `(2,3)` are both valid. `[100]` cannot produce a pair for `200`. `[100,100]` can.

**A — 6 minutes:** implement nested loops.
**B — 9 minutes:** improve using a dictionary of earlier prices and their indices.

Before coding, write: What do I need to know when I visit the current price?


In [ ]:
def budget_pair_brute(prices, budget):
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            if prices[i] + prices[j] == budget:
                return (i, j)
    return None


In [ ]:
def budget_pair(prices, budget):
    seen = {}
    for i, price in enumerate(prices):
        needed = budget - price
        if needed in seen:
            return (seen[needed], i)
        seen[price] = i
    return None


In [ ]:
def check_pair(fn, values, target):
    original = values[:]
    result = fn(values, target)
    exists = any(values[i] + values[j] == target
                 for i in range(len(values))
                 for j in range(i + 1, len(values)))
    assert values == original, "Do not mutate the input"
    if not exists:
        assert result is None, f"Expected None, got {result}"
    else:
        assert isinstance(result, tuple) and len(result) == 2
        i, j = result
        assert isinstance(i, int) and isinstance(j, int)
        assert 0 <= i < len(values) and 0 <= j < len(values)
        assert i != j, "One item cannot be used twice"
        assert values[i] + values[j] == target, "Incorrect sum"

pair_cases = [([], 200), ([100], 200), ([100, 100], 200),
              ([120, 80, 160, 40], 200), ([40, 60], 150),
              ([0, 0], 0), ([120, 50, 80, 40], 200)]


In [ ]:
for fn in (budget_pair_brute, budget_pair):
    for values, target in pair_cases:
        check_pair(fn, values[:], target)
print("Budget-pair checks passed")


<details><summary>Hint 1: the repeated work</summary>
For a price of 80 and a budget of 200, the missing price is 120. Have you already seen it?
</details>
<details><summary>Hint 2: what to remember</summary>
Store `price: index` in a dictionary. Check whether `budget - price` exists.
</details>
<details><summary>Hint 3: ordering matters</summary>
Check for the complement before inserting the current item. Otherwise a single 100 could incorrectly match itself.
</details>


**Reasoning:** before each iteration, the dictionary holds only earlier items. If the complement exists, the two distinct indices form a valid pair. If a valid pair exists, its later element finds an earlier complement.

Nested loops: worst-case O(n²) time, O(1) extra space. Dictionary: expected O(n) time, O(n) extra space. Hash operations are average O(1), with worse collision cases possible.

**Common mistakes:** inserting before checking; returning prices instead of indices; requiring one particular answer when several exist; returning None inside the loop.


## 2. Two pointers (15 minutes)
The input is now already sorted in nondecreasing order.

**Contract:** return any pair of distinct indices in this sorted input, or `None`. Do not sort or modify it. The caller guarantees sorted input. All other pair requirements stay the same.

For `[20,40,60,80,120]` and `100`, `(0,3)` or `(1,2)` is valid.

Start one pointer at each end. If the sum is too small, which pointer should move? Explain why the discarded endpoint cannot form a valid answer with any remaining partner.


In [ ]:
def sorted_pair(prices, budget):
    left, right = 0, len(prices) - 1
    while left < right:
        total = prices[left] + prices[right]
        if total == budget:
            return (left, right)
        if total < budget:
            left += 1
        else:
            right -= 1
    return None


In [ ]:
for values, target in pair_cases + [([20,40,60,80,120],100)]:
    check_pair(sorted_pair, sorted(values), target)
print("Two-pointer checks passed")


<details><summary>Hints</summary>

1. Use `left = 0`, `right = len(prices) - 1`.
2. Continue while `left < right`.
3. Too small: move left forward. Too large: move right backward.
4. If the loop ends, return None.
</details>

**Discussion:** what changes if the input is unsorted and you must return original indices?


**Why it works:** with a sum too small, every other remaining partner for the left value is no larger than the current right value. None can reach the target, so discard left. For a sum too large, every other remaining partner for the right value is at least the current left value, so discard right.

At least one pointer moves each unsuccessful iteration, so O(n) time and O(1) extra space. Sorting arbitrary input adds O(n log n) work and requires preserving original indices if the output contract needs them.

Common errors: `left <= right` permits reusing an index; moving the wrong pointer; forgetting a pointer move and looping forever.


## 3. Busiest consecutive hours (15 minutes)
**Contract:** input is a list of nonnegative integer request counts and an integer `k`. Return `(start_index, maximum_total)` for exactly `k` consecutive entries. On a tie, return the earliest start. Do not mutate the input. Raise `ValueError` when `k < 1` or `k > len(requests)`, including empty input.

`[12,8,15,20,6,10]`, `k=3` gives `(1,43)`. The index is zero-based, so these are hours 2–4.

Trace all window totals by hand first. Then implement the rolling sum. The instructor will demonstrate the baseline of summing each window separately.

How can you update 35 for `[12,8,15]` to the total for `[8,15,20]`?


In [ ]:
def busiest_hours_brute(requests, k):
    if k < 1 or k > len(requests):
        raise ValueError("k must be between 1 and len(requests)")
    best_start, best_total = 0, None
    for start in range(len(requests) - k + 1):
        total = 0
        for offset in range(k):
            total += requests[start + offset]
        if best_total is None or total > best_total:
            best_start, best_total = start, total
    return (best_start, best_total)


In [ ]:
def busiest_hours(requests, k):
    if k < 1 or k > len(requests):
        raise ValueError("k must be between 1 and len(requests)")
    current = 0
    for i in range(k):
        current += requests[i]
    best_start, best_total = 0, current
    for right in range(k, len(requests)):
        current += requests[right] - requests[right - k]
        if current > best_total:
            best_start = right - k + 1
            best_total = current
    return (best_start, best_total)


In [ ]:
window_cases = [([12,8,15,20,6,10],3,(1,43)),
                ([4,9,3,12,2],2,(2,15)),
                ([5,5,5],2,(0,10)),
                ([3,7,2],1,(1,7)),
                ([3,7,2],3,(0,12)),
                ([0,0],1,(0,0))]
for values, k, expected in window_cases:
    before = values[:]
    assert busiest_hours(values, k) == expected
    assert values == before
for values, k in [([],1), ([3],0), ([3],2), ([3],-1)]:
    try:
        busiest_hours(values, k)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid k should raise ValueError")
print("Sliding-window checks passed")


<details><summary>Hint 1: the first window</summary>
Sum the first k values once. Initialize your best answer from this real window.
</details>
<details><summary>Hint 2: moving one position</summary>
At incoming index `right`, add `requests[right]` and subtract `requests[right-k]`.
</details>
<details><summary>Hint 3: ties and indices</summary>
The new window starts at `right-k+1`. Update only when the new total is strictly greater to preserve the earliest tie.
</details>


**Invariant:** current always equals the sum of the current k entries. Subtracting the outgoing item and adding the incoming one preserves it. The best result stores the earliest maximum among all windows examined.

The rolling algorithm uses O(n) time and O(1) extra space. The initial loop avoids creating a Python slice. The baseline does O((n−k+1)k) work, upper bounded by O(nk). For fixed k=3, both methods are O(n); rolling avoids repeated additions.

**Common mistakes:** wrong outgoing index; missing the final window; using `>=` changes the tie rule; resetting the best total to zero rather than a real window.

**Extension:** allow negative entries and run `busiest_hours([-5,-2,-8],2)`, which returns `(0,-7)`. Fixed-size rolling sums still work. Some variable-window techniques need nonnegative values and do not transfer automatically.


## 4. Explain and reflect (7 minutes)
Choose one solution and answer:

1. What is the input/output contract?
2. Why is the solution correct? What remains true during the loop?
3. What repeated work did you remove?
4. What are its time and extra-space costs?
5. Which test caught a mistake, or could catch one?

Write your answers below before running more code.


**My explanation:**

- Contract:
- Reasoning:
- Repeated work removed:
- Time and extra space:
- Useful test:


## Assessment and optional extensions
Award one point each for: clear contract, correct implementation, explanation of why it works, meaningful edge cases, accurate complexity. Use the rubric for formative feedback.

Extensions: (1) return all matching index pairs and discuss output size; (2) allow negative request values; (3) compare the number of additions across window sizes; (4) preserve original indices when sorting. Avoid pushing beginners into variable windows before they can explain a fixed window.

The following deterministic randomized checks compare solutions with a baseline. They are instructor checks, not required student work.


In [ ]:
import random
rng = random.Random(42)
for _ in range(250):
    values = [rng.randrange(0, 30) for _ in range(rng.randrange(0, 15))]
    target = rng.randrange(0, 60)
    for fn in (budget_pair_brute, budget_pair):
        check_pair(fn, values[:], target)
    check_pair(sorted_pair, sorted(values), target)
for _ in range(250):
    values = [rng.randrange(-20, 40) for _ in range(rng.randrange(1, 20))]
    k = rng.randrange(1, len(values) + 1)
    assert busiest_hours(values,k) == busiest_hours_brute(values,k)
assert busiest_hours([-5,-2,-8],2) == (0,-7)
print("All instructor comparison checks passed")
